# 8. Gán người chơi và đề xuất chiến lược

Nếu thiếu tệp trung gian, chạy lần lượt notebook 04 đến phần ngay trước file này.

Muốn chạy toàn bộ trong một kernel: [mở notebook 00](00_Gamer_Behavior_Clustering.ipynb).

<details><summary>Danh sách tám phần</summary>

- [1. Tìm hiểu bài toán và dữ liệu](01_data_understanding.ipynb)

- [2. Thống kê và khám phá dữ liệu (EDA)](02_statistics_eda.ipynb)

- [3. Kiểm tra và làm sạch dữ liệu](03_data_cleaning.ipynb)

- [4. Chia dữ liệu và chuẩn hóa](04_data_transformation.ipynb)

- [5. Ward, K-Means và cách kết hợp](05_clustering.ipynb)

- [6. Đánh giá holdout và độ nhạy](06_model_evaluation.ipynb)

- [7. Hồ sơ người chơi và trực quan hóa](07_player_profiling.ipynb)

- [8. Gán người chơi và đề xuất chiến lược](08_prediction_strategy.ipynb)

</details>

## Khởi tạo và đọc dữ liệu

In [1]:
from pathlib import Path
import os, sys, json, hashlib
import numpy as np
import pandas as pd
import joblib
from IPython.display import display, Markdown

# Tìm gốc dự án khi mở từ thư mục notebooks/, gamer_profiling/ hoặc workspace.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "gamer_profiling"]
ROOT = next((p for p in candidates if (p / "gamer_clustering.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án gamer_profiling hoặc notebooks/.")
NBOUT = Path(os.environ.get("GAMER_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebook/latest")))
NBOUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(NBOUT / ".matplotlib"))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from gamer_clustering import clean_features, centers_from_labels, metrics, choose_k
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (pairwise_distances, silhouette_score, silhouette_samples,
                             davies_bouldin_score, adjusted_rand_score)
from sklearn.metrics.pairwise import pairwise_distances_argmin
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, cut_tree, dendrogram
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
import seaborn as sns

# Module .py dùng Agg khi chạy batch; trong notebook cần hiển thị hình ngay dưới ô code.
get_ipython().run_line_magic("matplotlib", "inline")
thread_limit = threadpool_limits(limits=2)
sns.set_theme(style="whitegrid", font="DejaVu Sans", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)
SEED = 42
FEATURES = ["PlayTimeHours", "SessionsPerWeek", "AvgSessionDurationMinutes",
            "PlayerLevel", "AchievementsUnlocked"]
METHODS = ["KMeans", "Ward", "Ward_KMeans"]
SOURCE = ROOT / "data/online_gaming_behavior_dataset.csv"
df = pd.read_csv(SOURCE)
reference = json.loads((ROOT / "outputs/summary.json").read_text(encoding="utf-8"))

def require(name, previous):
    path = NBOUT / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}. Chạy notebook {previous} trước, hoặc Run All notebook 00.")
    return path

def save_plot(name):
    plt.tight_layout()
    plt.savefig(NBOUT / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

Cần notebook 07 để có mô hình và tên hồ sơ. Với dữ liệu mới chỉ `transform` rồi `predict`;
không fit lại scaler trên người mới. Ví dụ dưới lấy năm dòng thật của tập holdout,
không tạo hồ sơ giả để đưa vào kết quả.

In [2]:
artifact = joblib.load(require("player_model.joblib", "07_player_profiling.ipynb"))
state = joblib.load(require("preprocessing.joblib", "04_data_transformation.ipynb"))
examples = df.iloc[state["test_idx"][:5]][["PlayerID"] + FEATURES].copy()
examples["cluster"] = artifact["model"].predict(artifact["prep"].transform(clean_features(examples)))
examples["profile"] = examples.cluster.map(artifact["cluster_names"])
display(examples)
examples.to_csv(NBOUT / "prediction_examples.csv", index=False, encoding="utf-8-sig")

,PlayerID,PlayTimeHours,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,cluster,profile
34667,43667,3.3269,16,63,43,6,6,"Chơi ít phiên, phiên ngắn"
22227,31227,19.6265,10,99,48,11,3,"Chơi thường xuyên, phiên ngắn"
24899,33899,23.3221,11,17,12,34,7,"Chơi thường xuyên, phiên ngắn"
35415,44415,10.3488,7,149,44,38,1,"Chơi thường xuyên, phiên dài"
21440,30440,6.2387,10,127,14,4,6,"Chơi ít phiên, phiên ngắn"


**Chiến lược là giả thuyết:** không có tiền mua, chi phí hoặc can thiệp nên chưa tính ROI/uplift.
Có thể thử trang trí nhỏ cho phiên ngắn, nội dung mở rộng cho phiên dài, điểm chạm khi quay lại
cho nhóm ít phiên. Không suy ra mức sẵn sàng chi trả từ cấp độ hoặc thành tích.

In [3]:
profile = pd.read_csv(require("cluster_profiles.csv", "07_player_profiling.ipynb"))
strategies = []
for row in profile.itertuples():
    short = "phiên ngắn" in row.profile
    frequent = "thường xuyên" in row.profile
    strategies.append({"Cụm": row.cluster, "Hồ sơ": row.profile,
                       "Giả thuyết nội dung": "Gói trang trí nhỏ, dễ trải nghiệm" if short else "Nội dung mở rộng, trang trí theo chủ đề",
                       "Điểm chạm": "Kết thúc phiên; giới hạn nhắc mua" if frequent else "Khi quay lại game; nhiệm vụ ngắn",
                       "Kiểm chứng": "A/B test: conversion, doanh thu/người, retention, hoàn tiền"})
strategy = pd.DataFrame(strategies)
display(strategy)
strategy.to_csv(NBOUT / "strategies.csv", index=False, encoding="utf-8-sig")

,Cụm,Hồ sơ,Giả thuyết nội dung,Điểm chạm,Kiểm chứng
0,0,"Chơi ít phiên, phiên ngắn","Gói trang trí nhỏ, dễ trải nghiệm",Khi quay lại game; nhiệm vụ ngắn,"A/B test: conversion, doanh thu/người, retenti..."
1,1,"Chơi thường xuyên, phiên dài","Nội dung mở rộng, trang trí theo chủ đề",Kết thúc phiên; giới hạn nhắc mua,"A/B test: conversion, doanh thu/người, retenti..."
2,2,"Chơi ít phiên, phiên dài","Nội dung mở rộng, trang trí theo chủ đề",Khi quay lại game; nhiệm vụ ngắn,"A/B test: conversion, doanh thu/người, retenti..."
3,3,"Chơi thường xuyên, phiên ngắn","Gói trang trí nhỏ, dễ trải nghiệm",Kết thúc phiên; giới hạn nhắc mua,"A/B test: conversion, doanh thu/người, retenti..."
4,4,"Chơi thường xuyên, phiên dài","Nội dung mở rộng, trang trí theo chủ đề",Kết thúc phiên; giới hạn nhắc mua,"A/B test: conversion, doanh thu/người, retenti..."
5,5,"Chơi ít phiên, phiên dài","Nội dung mở rộng, trang trí theo chủ đề",Khi quay lại game; nhiệm vụ ngắn,"A/B test: conversion, doanh thu/người, retenti..."
6,6,"Chơi ít phiên, phiên ngắn","Gói trang trí nhỏ, dễ trải nghiệm",Khi quay lại game; nhiệm vụ ngắn,"A/B test: conversion, doanh thu/người, retenti..."
7,7,"Chơi thường xuyên, phiên ngắn","Gói trang trí nhỏ, dễ trải nghiệm",Kết thúc phiên; giới hạn nhắc mua,"A/B test: conversion, doanh thu/người, retenti..."


Phân ngẫu nhiên control/treatment **trong từng cụm**, khóa nhãn trước thử nghiệm, đo trên tất cả
người đủ điều kiện được phân nhánh. Cần bổ sung giao dịch có số tiền/thời điểm, wins/losses,
vật phẩm theo thời gian và MMR/Elo trước khi mở rộng diễn giải.

Cuối cùng kiểm chứng notebook với bộ kết quả đã dùng trong báo cáo. Hai cách chạy phải khớp
bảng chọn k, điểm holdout, hồ sơ, độ ổn định và gán người mới trong sai số số học cho phép.

In [4]:
numerical_columns = ["silhouette", "davies_bouldin", "min_cluster_pct"]
checks = []
for filename, keys, values in [
    ("model_selection.csv", ["method", "k"], numerical_columns),
    ("holdout_metrics.csv", ["method", "k"], numerical_columns),
    ("stability.csv", ["seed"], ["ari_vs_seed42"] + numerical_columns),
    ("cluster_profiles.csv", ["cluster"], FEATURES + ["count", "share_pct", "purchase_rate"]),
]:
    new = pd.read_csv(require(filename, "04 → 07")).sort_values(keys).reset_index(drop=True)
    old = pd.read_csv(ROOT / "outputs" / filename).sort_values(keys).reset_index(drop=True)
    pd.testing.assert_frame_equal(new[keys], old[keys], check_dtype=False)
    np.testing.assert_allclose(new[values].to_numpy(float), old[values].to_numpy(float), rtol=1e-7, atol=1e-9)
    checks.append({"Kiểm tra": filename, "Kết quả": "Khớp kết quả báo cáo"})
original_examples = pd.read_csv(ROOT / "outputs/prediction_examples.csv")
np.testing.assert_array_equal(examples.cluster, original_examples.cluster)
all_predictions = artifact["model"].predict(artifact["prep"].transform(clean_features(df)))
original_labels = pd.read_csv(ROOT / "outputs/player_assignments.csv").cluster.to_numpy()
assert adjusted_rand_score(all_predictions, original_labels) == 1.0
checks.append({"Kiểm tra": "Gán nhóm toàn bộ 40.034 người và ví dụ mới", "Kết quả": "Khớp mô hình đã nộp"})
sensitivity = json.loads((NBOUT / "feature_sensitivity.json").read_text(encoding="utf-8"))
for key in ["ari_vs_main", "silhouette", "davies_bouldin"]:
    np.testing.assert_allclose(sensitivity[key], reference["feature_sensitivity"][key], rtol=1e-7, atol=1e-9)
checks.append({"Kiểm tra": "Bỏ PlayTimeHours", "Kết quả": "Khớp phân tích độ nhạy"})
display(pd.DataFrame(checks))
final = pd.read_csv(NBOUT / "holdout_metrics.csv").set_index("method").loc["Ward_KMeans"]
display(Markdown(f"**Kết quả chạy:** k = {int(final.k)} trong dải 2–8; "
                 f"Silhouette holdout = **{final.silhouette:.4f}**; DBI = **{final.davies_bouldin:.4f}**. "
                 "Ranh giới còn chồng lấn; dữ liệu tổng hợp và độ nhạy mẫu giới hạn khả năng áp dụng."))
(NBOUT / "verification.json").write_text(json.dumps({"passed": True, "checks": checks},
                                                    ensure_ascii=False, indent=2), encoding="utf-8")

,Kiểm tra,Kết quả
0,model_selection.csv,Khớp kết quả báo cáo
1,holdout_metrics.csv,Khớp kết quả báo cáo
2,stability.csv,Khớp kết quả báo cáo
3,cluster_profiles.csv,Khớp kết quả báo cáo
4,Gán nhóm toàn bộ 40.034 người và ví dụ mới,Khớp mô hình đã nộp
5,Bỏ PlayTimeHours,Khớp phân tích độ nhạy


**Kết quả chạy:** k = 8 trong dải 2–8; Silhouette holdout = **0.1540**; DBI = **1.4322**. Ranh giới còn chồng lấn; dữ liệu tổng hợp và độ nhạy mẫu giới hạn khả năng áp dụng.

618